# M1 clean-clone rerun (exit criterion 6)

A second team member reruns part of the M1 skeleton **from a fresh clone, on their own Colab account**, with nothing from the first member's Drive: code from GitHub, llama.cpp built from source at the pinned commit, model files rebuilt from Hugging Face.

The check (`scripts/clean_clone_check.py`) passes when:
1. the dry run expands to the same 253 specs as the Oct 25 run (experiment hash `3e3a25e70f`, same 14 configuration keys);
2. 10 episodes of BFCL Q4_0 on CUDA run end to end, every step logged and classified;
3. the verdicts agree with the reference (that configuration scored 0 of 40).

Whether the rebuilt model files match the pilot's SHA-256 is reported too (a reproducibility finding, not a pass condition).

* Runtime: **T4** is enough (cheapest, 1.2 compute units/h); L4 also works. About 40 minutes, most of it building llama.cpp.
* Send the last cell's JSON (or `/content/cc_runs/clean_clone_check.json`) to the team.


In [ ]:
# 1. Fresh clone (no Drive, nothing cached)
!rm -rf /content/cmpe_295a && git clone -q -b m1/skeleton https://github.com/VritikaMalhotra/cmpe_295a /content/cmpe_295a
%cd /content/cmpe_295a
!git log -1 --oneline
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Environment + llama.cpp built from source at the pilot's commit (CPU and CUDA builds; ~25 min)
!bash scripts/setup_env.sh --llama --cuda

In [ ]:
# 3. Model files rebuilt from Hugging Face with the pinned converter: BF16, then pure Q4_0 (~5 min)
!env -u UV_SYSTEM_PYTHON uv pip install -q --python .venv/bin/python -e ".[train]"
!bash scripts/make_probe_models.sh Qwen/Qwen3-0.6B /content/models

In [ ]:
# 4. The check: same plan, 10 episodes, same verdicts
import os
os.environ.update(TESTBED_MODELS='/content/models',
                  LLAMA_CUDA_BIN='/content/cmpe_295a/llama.cpp/build-cuda/bin/llama-server',
                  LLAMA_CPU_BIN='/content/cmpe_295a/llama.cpp/build-cpu/bin/llama-server')
!.venv/bin/python scripts/clean_clone_check.py --out /content/cc_runs --limit 10

In [ ]:
# 5. Release this runtime (copy the JSON above first). Set RELEASE = False to keep it.
RELEASE = True
if RELEASE:
    from google.colab import runtime
    runtime.unassign()